In [11]:
TRAIN_PATH = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv"   # 파일 형식에 맞게 read_csv / read_parquet 등으로 수정
TEST_PATH = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_unlabeled_cn7.csv"
TARGET = "passorfail"

In [12]:
"""
train/test가 같은 scaler로 표준화됐는지, 분포 모양이 같은지 확인하는 진단 스크립트.
제품(cn7, rg3)별로 따로 실행하세요. 경로와 컬럼명만 바꾸면 됩니다.
"""
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# ===== 설정 =====
TRAIN_PATH = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv"   # 파일 형식에 맞게 read_csv / read_parquet 등으로 수정
TEST_PATH = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_unlabeled_cn7.csv"
TARGET = "passorfail"
DROP_COLS = []                 # id, 시간 등 숫자지만 피처가 아닌 컬럼이 있으면 추가
SEED = 42
# ================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

cols = [
    c for c in train.columns
    if c in test.columns and c != TARGET and c not in DROP_COLS
    and pd.api.types.is_numeric_dtype(train[c])
]
tr, te = train[cols].copy(), test[cols].copy()
print(f"공통 숫자 컬럼 {len(cols)}개 | train {tr.shape} | test {te.shape}\n")

# 1) 평균/표준편차 (표준화 확인)
print("=== 1) 평균 / 표준편차 ===")
summary = pd.DataFrame({
    "tr_mean": tr.mean(), "tr_std": tr.std(),
    "te_mean": te.mean(), "te_std": te.std(),
})
print(summary.round(3), "\n")

# 2) 이산값 컬럼: z값 레벨이 같은지 / 선형 관계(a*z+b)로 어긋나는지
print("=== 2) 이산값 컬럼 레벨 비교 (고유값 30개 이하) ===")
for c in cols:
    u_tr = np.sort(tr[c].dropna().round(4).unique())
    u_te = np.sort(te[c].dropna().round(4).unique())
    if len(u_tr) > 30 or len(u_te) > 30:
        continue
    if len(u_tr) == len(u_te) and len(u_tr) >= 3:
        a, b = np.polyfit(u_tr, u_te, 1)
        resid = np.abs(np.polyval([a, b], u_tr) - u_te).max()
        print(f"{c}: 레벨 {len(u_tr)}개 | te ≈ {a:.3f}*tr + {b:.3f} (최대 잔차 {resid:.4f})")
    else:
        print(f"{c}: 고유값 수 다름 (train {len(u_tr)}, test {len(u_te)})")
print("-> a≈1, b≈0 이면 같은 스케일러, 아니면 독립 표준화 가능성\n")

# 3) 분위수 / KS 통계량
print("=== 3) 분위수 및 KS 통계량 ===")
qs = [0.05, 0.25, 0.5, 0.75, 0.95]
rows = []
for c in cols:
    q_tr = tr[c].quantile(qs).values
    q_te = te[c].quantile(qs).values
    ks = stats.ks_2samp(tr[c].dropna(), te[c].dropna()).statistic
    rows.append([c, *np.round(q_tr, 2), *np.round(q_te, 2), round(ks, 3)])
qdf = pd.DataFrame(
    rows,
    columns=["col"] + [f"tr_q{int(q*100)}" for q in qs] + [f"te_q{int(q*100)}" for q in qs] + ["KS"],
)
print(qdf.sort_values("KS", ascending=False).to_string(index=False), "\n")

# 4) 왜도 / 첨도 (아핀 변환에 불변)
print("=== 4) 왜도 / 첨도 ===")
shape = pd.DataFrame({
    "tr_skew": tr.skew(), "te_skew": te.skew(),
    "tr_kurt": tr.kurt(), "te_kurt": te.kurt(),
})
shape["skew_diff"] = (shape.tr_skew - shape.te_skew).abs()
print(shape.sort_values("skew_diff", ascending=False).round(3), "\n")

# 5) 상관행렬 차이
print("=== 5) 상관행렬 차이 ===")
diff = tr.corr() - te.corr()
print(f"Frobenius norm: {np.linalg.norm(diff.fillna(0).values):.3f}")
print(f"최대 절대 차이: {diff.abs().max().max():.3f}\n")

# 6) Adversarial validation (train vs test 구분 가능한가)
print("=== 6) Adversarial validation ===")
rng = np.random.RandomState(SEED)
te_s = te.sample(n=min(len(te), 5000), random_state=SEED)
X = pd.concat([tr, te_s]).fillna(0).reset_index(drop=True)
y = np.r_[np.zeros(len(tr)), np.ones(len(te_s))]
rf = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, class_weight="balanced",
    n_jobs=-1, random_state=SEED,
)
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
proba = cross_val_predict(rf, X, y, cv=skf, method="predict_proba")[:, 1]
print(f"AUC: {roc_auc_score(y, proba):.3f}  (0.5 근처면 같은 분포, 높을수록 분포 차이 큼)")
rf.fit(X, y)
imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
print("구분에 기여한 상위 변수:")
print(imp.head(10).round(4))

공통 숫자 컬럼 25개 | train (1211, 25) | test (35239, 25)

=== 1) 평균 / 표준편차 ===
                          tr_mean  tr_std     te_mean     te_std
Unnamed: 0                  605.0  349.73  638822.197  163261.31
Injection_Time               -0.0    1.00       0.000       1.00
Filling_Time                 -0.0    1.00      -0.000       1.00
Plasticizing_Time             0.0    1.00      -0.000       1.00
Cycle_Time                    0.0    1.00       0.000       1.00
Clamp_Close_Time             -0.0    1.00      -0.000       1.00
Cushion_Position              0.0    1.00      -0.000       1.00
Plasticizing_Position        -0.0    1.00      -0.000       1.00
Clamp_Open_Position           0.0    0.00       0.000       1.00
Max_Injection_Speed           0.0    1.00      -0.000       1.00
Max_Screw_RPM                -0.0    1.00       0.000       1.00
Average_Screw_RPM             0.0    1.00       0.000       1.00
Max_Injection_Pressure        0.0    1.00      -0.000       1.00
Max_Switch_Over_P

test의 많은 변수가 "두 수준"으로 갈라져 있어. Cushion_Position, Average_Screw_RPM, Barrel_Temperature_1, Average_Back_Pressure, Mold_Temperature_3·4 등이 q5~q50이 −0.97 근처, q75~q95가 +1.03 근처이고 첨도가 −2에 가까워. 값이 두 덩어리로 나뉘고 비율이 대략 50:50이라는 뜻이야. 여러 변수가 동시에 그러니 설정이나 운전 조건이 다른 두 레짐이 섞여 있다고 추정돼. train도 Cushion_Position이 −0.71/+1.20으로 갈라지는데 비율은 약 63:37이야. 즉 같은 두 레짐이 섞여 있지만 비율이 다르고, 표준화가 따로 되어 z값 위치가 달라진 것일 가능성이 커.
train은 극단값 몇 개가 표준편차를 키우고 있어. Plasticizing_Time 왜도 15, 첨도 281 같은 값은 소수 행이 극단적이라는 뜻이야. 그 결과 train의 나머지 행은 z 기준으로 좁은 구간에 눌려 있고, test는 레짐 분리 때문에 넓게 퍼져서, 같은 z값이라도 train과 test에서 실제 크기가 몇 배씩 달라질 수 있어. 이 극단 행이 불량 17개와 겹치는지 아직 몰라.
상관행렬 차이가 크고(Frobenius 19.2, 최대 1.86) test에는 왼쪽 꼬리가 길어. Hopper_Temperature(왜도 −2.96), Barrel_Temperature_4(−2.74)는 test에 낮은 온도 구간(예열이나 재가동 같은 상태)이 따로 있다는 신호일 수 있어.

In [15]:
"""
2차 진단 스크립트 (제품별로 따로 실행: cn7, rg3)
  A) train 극단값 행과 불량의 관계
  B) 인덱스/상수 컬럼을 제거한 adversarial validation
  C) test의 '2수준(레짐)' 구조를 train과 비교
  D) 레짐별 robust 정규화 후 train/test 차이가 줄어드는지 확인
"""
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# ===== 설정 =====
TRAIN_PATH = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv"   # 파일 형식에 맞게 read_csv / read_parquet 등으로 수정
TEST_PATH = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_unlabeled_cn7.csv"
TARGET = "passorfail"
TARGET = "passorfail"       # 대소문자 무시하고 자동 탐색 (예: PassOrFail)
DEFECT_VALUE = 1            # 불량을 뜻하는 값 (0/1이 아니라 'N', 'Fail' 등이면 여기 수정)
DROP_COLS = ["Unnamed: 0"]  # 인덱스 컬럼은 반드시 제외
# test에서 2수준으로 갈라지던 변수들 (3번 결과 기준)
REGIME_COLS = [
    "Cushion_Position", "Plasticizing_Position", "Average_Screw_RPM",
    "Max_Screw_RPM", "Clamp_Close_Time",
]
SEED = 42
# ================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

cand = [c for c in train.columns if c.lower() == TARGET.lower()]
if not cand:
    raise SystemExit(f"타깃 컬럼을 찾지 못했습니다. train 컬럼 목록:\n{list(train.columns)}")
TARGET = cand[0]
print(f"타깃 컬럼: {TARGET} | 고유값: {train[TARGET].unique()[:10]}")

cols = [
    c for c in train.columns
    if c in test.columns and c != TARGET and c not in DROP_COLS
    and pd.api.types.is_numeric_dtype(train[c])
]
const = [c for c in cols if train[c].std() < 1e-9]
print(f"train에서 상수인 컬럼 (제외): {const}")
cols = [c for c in cols if c not in const]
tr, te = train[cols].copy(), test[cols].copy()
y_tr = (train[TARGET] == DEFECT_VALUE).astype(int).values
print(f"사용 컬럼 {len(cols)}개\n")


def adversarial(a, b, label, n_test=5000):
    b = b.sample(n=min(len(b), n_test), random_state=SEED)
    X = pd.concat([a, b]).fillna(0).reset_index(drop=True)
    y = np.r_[np.zeros(len(a)), np.ones(len(b))]
    rf = RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced",
        n_jobs=-1, random_state=SEED,
    )
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    p = cross_val_predict(rf, X, y, cv=skf, method="predict_proba")[:, 1]
    auc = roc_auc_score(y, p)
    rf.fit(X, y)
    imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print(f"[{label}] adversarial AUC = {auc:.3f}")
    print(imp.head(5).round(3).to_string(), "\n")
    return auc


# A) 극단값과 불량의 관계
print("=== A) train 극단값 행 vs 불량 ===")
maxz = tr.abs().max(axis=1)
for lo, hi in [(0, 3), (3, 5), (5, np.inf)]:
    m = (maxz > lo) & (maxz <= hi)
    n = int(m.sum())
    d = int(y_tr[m.values].sum())
    rate = d / n if n else float("nan")
    print(f"max|z| in ({lo}, {hi}]: 행 {n}개, 불량 {d}개, 불량률 {rate:.3f}")
print(f"전체 불량 {int(y_tr.sum())}개 중 max|z|>4 에 속하는 불량: "
      f"{int(y_tr[(maxz > 4).values].sum())}개\n")

# B) 인덱스/상수 제거 후 adversarial
print("=== B) 인덱스/상수 제거 후 adversarial validation ===")
adversarial(tr, te, "원본 z값")

# C) 레짐 구조 비교
print("=== C) 레짐(KMeans k=2) 구조 비교 ===")
rc = [c for c in REGIME_COLS if c in cols]


def winsor_z(X):
    Xc = X.clip(X.quantile(0.01), X.quantile(0.99), axis=1)
    sd = Xc.std().replace(0, 1)
    return (Xc - Xc.mean()) / sd


def fit_regime(X, name):
    Z = winsor_z(X[rc])
    km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(Z)
    lab = km.labels_
    # 첫 번째 변수 중심값이 작은 쪽을 0으로 통일
    order = np.argsort(km.cluster_centers_[:, 0])
    remap = {order[0]: 0, order[1]: 1}
    lab = np.array([remap[v] for v in lab])
    idx = np.random.RandomState(SEED).choice(len(Z), min(len(Z), 3000), replace=False)
    sil = silhouette_score(Z.iloc[idx], lab[idx])
    print(f"[{name}] 레짐 비율 = {np.bincount(lab) / len(lab)} | silhouette = {sil:.3f}")
    print(pd.DataFrame(km.cluster_centers_[order], columns=rc).round(2).to_string())
    return lab


lab_tr = fit_regime(tr, "train")
lab_te = fit_regime(te, "test")
for r in [0, 1]:
    m = lab_tr == r
    print(f"train 레짐 {r}: 행 {int(m.sum())}개, 불량 {int(y_tr[m].sum())}개, "
          f"불량률 {y_tr[m].mean():.4f}")
print()


# D) 레짐별 robust 정규화
def regime_robust(X, lab):
    out = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
    for r in np.unique(lab):
        m = lab == r
        sub = X[m]
        med = sub.median()
        mad = 1.4826 * (sub - med).abs().median()
        scale = mad.where(mad > 1e-9, sub.std())
        scale = scale.where(scale > 1e-9, 1.0)
        out.loc[m] = ((sub - med) / scale).values
    return out


print("=== D) 레짐별 robust 정규화 후 adversarial ===")
tr_rn = regime_robust(tr, lab_tr)
te_rn = regime_robust(te, lab_te)
adversarial(tr_rn, te_rn, "레짐별 robust 정규화")

tr_rn.assign(regime=lab_tr, **{TARGET: y_tr}).to_csv("train_regime_norm.csv", index=False)
te_rn.assign(regime=lab_te).to_csv("test_regime_norm.csv", index=False)
print("저장: train_regime_norm.csv, test_regime_norm.csv")

타깃 컬럼: PassOrFail | 고유값: [0 1]
train에서 상수인 컬럼 (제외): ['Clamp_Open_Position']
사용 컬럼 23개

=== A) train 극단값 행 vs 불량 ===
max|z| in (0, 3]: 행 1163개, 불량 11개, 불량률 0.009
max|z| in (3, 5]: 행 32개, 불량 0개, 불량률 0.000
max|z| in (5, inf]: 행 16개, 불량 6개, 불량률 0.375
전체 불량 17개 중 max|z|>4 에 속하는 불량: 6개

=== B) 인덱스/상수 제거 후 adversarial validation ===
[원본 z값] adversarial AUC = 1.000
Plasticizing_Time         0.212
Average_Screw_RPM         0.125
Cushion_Position          0.108
Max_Injection_Pressure    0.094
Plasticizing_Position     0.085 

=== C) 레짐(KMeans k=2) 구조 비교 ===
[train] 레짐 비율 = [0.62510322 0.37489678] | silhouette = 0.350
   Cushion_Position  Plasticizing_Position  Average_Screw_RPM  Max_Screw_RPM  Clamp_Close_Time
0             -0.76                   0.33               0.02           0.09             -0.48
1              1.27                  -0.56              -0.03          -0.15              0.80
[test] 레짐 비율 = [0.51508272 0.48491728] | silhouette = 0.970
   Cushion_Position  Plasticizing_Positi

1. 불량 17개 중 11개는 단순 이상치가 아니야 (A).

극단값 행(max|z|>5) 16개 중 불량이 6개야. 불량률 37.5%로 평균(1.4%)의 약 27배라 단순 이상치 탐지로도 그 6개는 잡혀.
나머지 불량 11개는 단변량 z값으로는 전부 정상 범위 안에 있어. 단순 이상치 탐지로 잡을 수 있는 recall 상한은 대략 35%야.
이 11개를 얼마나 잡느냐가 이 대회의 승부처이고, 오류분석 파트의 핵심 소재이기도 해.

2. 불량이 레짐에 크게 쏠려 있어 (C).

train에서 Cushion_Position이 낮은 쪽 레짐(전체의 약 62%)에 불량 17개 중 16개가 있어. 불량률이 2.1% 대 0.2%로, 대략 계산해도 우연일 가능성이 1% 아래야. 이건 가장 강한 신호야.
다만 이 레짐이 test의 어느 쪽과 대응되는지는 지금 데이터로는 확인할 수 없어.

3. train과 test의 레짐 구조가 달라 (C).

train은 레짐 분리가 약해(silhouette 0.35). 스크류 RPM 계열은 레짐 구분에 기여하지 않고, Cushion과 Plasticizing_Position은 오히려 반대 방향으로 움직여.
test는 5개 변수가 완벽하게 동시에 갈라져(silhouette 0.97).
그래서 "같은 두 레짐이 비율만 다르다"는 내 가설은 기각해야 해. test는 설정이 다른 두 운전 조건이 반반 섞인 데이터로 보이는데, 이건 추정이고 단정할 근거는 없어.

4. 정규화로 두 분포를 맞추려는 시도는 실패했어 (D).

레짐별 정규화 후에도 adversarial AUC가 1.000이야. 인덱스를 뺀 뒤에도 그래. 모델을 train에서 test로 옮기는 전략은 접어야 해.
이건 보고서에 쓸 중요한 발견이야. "train 점수로 test 성능을 보증할 수 없다"는 사실이 곧 진단 결과니까.

In [16]:
"""
3차 스크립트 (제품별로 따로 실행: cn7, rg3)
모델을 train -> test로 옮기지 않고 '절차'를 옮긴다.
  1) 각 데이터셋을 Cushion_Position 기준 2개 레짐으로 나눈다.
  2) 레짐 안에서 robust 정규화 후 비지도 이상 점수(IsolationForest / PCA 재구성 / Mahalanobis)를 만든다.
  3) 같은 절차를 train에 적용해 불량 17개를 얼마나 잡는지 검증한다.
  4) 같은 절차를 test(자기 자신으로 fit)에 적용해 점수를 저장한다.
  5) 비교용으로 train 지도학습(반복 CV) 성능도 출력한다.
"""
import warnings
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, rankdata
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf, MinCovDet
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore")

# ===== 설정 =====

TARGET = "PassOrFail"
DEFECT_VALUE = 1
DROP_COLS = ["Unnamed: 0"]
REGIME_COL = "Cushion_Position"
OUT_PATH = "test_anomaly_scores.csv"
SEED = 42
# ================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = [c for c in train.columns if c.lower() == TARGET.lower()][0]
y = (train[TARGET] == DEFECT_VALUE).astype(int).values

cols = [
    c for c in train.columns
    if c in test.columns and c != TARGET and c not in DROP_COLS
    and pd.api.types.is_numeric_dtype(train[c]) and train[c].std() > 1e-9
]
tr, te = train[cols].copy(), test[cols].copy()
print(f"사용 컬럼 {len(cols)}개 | train {tr.shape} 불량 {y.sum()}개 ({y.mean():.3%}) | test {te.shape}\n")


def get_regime(x):
    v = x.clip(x.quantile(0.01), x.quantile(0.99)).values.reshape(-1, 1)
    km = KMeans(2, n_init=10, random_state=SEED).fit(v)
    lab = km.labels_
    if km.cluster_centers_[0, 0] > km.cluster_centers_[1, 0]:
        lab = 1 - lab
    return lab


def robust_norm(X, lab):
    out = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
    for r in np.unique(lab):
        m = lab == r
        sub = X[m]
        med = sub.median()
        mad = 1.4826 * (sub - med).abs().median()
        scale = mad.where(mad > 1e-9, sub.std())
        scale = scale.where(scale > 1e-9, 1.0)
        out.loc[m] = ((sub - med) / scale).values
    return out


def zr(s):
    med = np.median(s)
    mad = 1.4826 * np.median(np.abs(s - med))
    if mad < 1e-9:
        mad = s.std() if s.std() > 1e-9 else 1.0
    return (s - med) / mad


def anomaly_scores(X, lab):
    Xn = robust_norm(X, lab)
    out = {k: np.zeros(len(X)) for k in ["iforest", "pca", "mahal"]}
    for r in np.unique(lab):
        m = lab == r
        A = Xn[m].values
        A = A[:, A.std(axis=0) > 1e-9]
        Ac = np.clip(A, -15, 15)
        iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(A)
        out["iforest"][m] = zr(-iso.score_samples(A))
        k = max(1, min(5, A.shape[1] - 1))
        pca = PCA(n_components=k, random_state=SEED).fit(Ac)
        rec = pca.inverse_transform(pca.transform(Ac))
        out["pca"][m] = zr(((Ac - rec) ** 2).sum(axis=1))
        try:
            idx = np.random.RandomState(SEED).choice(len(Ac), min(len(Ac), 5000), replace=False)
            cov = MinCovDet(support_fraction=0.9, random_state=SEED).fit(Ac[idx])
        except Exception:
            cov = LedoitWolf().fit(Ac)
        out["mahal"][m] = zr(cov.mahalanobis(Ac))
    out["ensemble"] = np.mean([rankdata(v) / len(v) for v in list(out.values())], axis=0)
    return out, Xn


def evaluate(y, s):
    order = np.argsort(-s)
    res = {"PR-AUC": average_precision_score(y, s), "ROC-AUC": roc_auc_score(y, s)}
    for frac in [0.02, 0.05, 0.10]:
        k = int(len(y) * frac)
        res[f"recall@{int(frac * 100)}%"] = y[order[:k]].sum() / y.sum()
    return res


# 1) 레짐
lab_tr, lab_te = get_regime(tr[REGIME_COL]), get_regime(te[REGIME_COL])
print("=== 1) 레짐 (기준 변수: %s) ===" % REGIME_COL)
print(f"train 레짐 비율: {np.bincount(lab_tr) / len(lab_tr)} | test 레짐 비율: {np.bincount(lab_te) / len(lab_te)}")
tab = [[int(y[lab_tr == r].sum()), int((lab_tr == r).sum() - y[lab_tr == r].sum())] for r in [0, 1]]
for r in [0, 1]:
    print(f"train 레짐 {r}: 행 {sum(tab[r])}개, 불량 {tab[r][0]}개 ({tab[r][0] / sum(tab[r]):.2%})")
print(f"Fisher 정확검정 p = {fisher_exact(tab)[1]:.4f}\n")

# 2) train에서 비지도 절차 검증
print("=== 2) 비지도 절차: train에서 검증 (기준선 PR-AUC = 불량률 %.3f) ===" % y.mean())
sc_tr, Xn_tr = anomaly_scores(tr, lab_tr)
print(pd.DataFrame({k: evaluate(y, v) for k, v in sc_tr.items()}).T.round(3).to_string())
ens_rank = rankdata(-sc_tr["ensemble"])
print("앙상블 기준 불량 17개의 순위(작을수록 잘 잡음):", sorted(ens_rank[y == 1].astype(int).tolist()), "\n")

# 3) 비교: 지도학습 반복 CV (레짐별 robust 정규화 피처 + 레짐)
print("=== 3) 비교: train 지도학습 (5-fold x 10회 반복 OOF PR-AUC) ===")
Xs = np.clip(Xn_tr.values, -15, 15)
Xs = np.c_[Xs, lab_tr]
models = {
    "logreg": LogisticRegression(C=0.1, class_weight="balanced", max_iter=3000),
    "extratrees": ExtraTreesClassifier(
        n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample",
        n_jobs=-1, random_state=SEED),
}
for name, mdl in models.items():
    aps = []
    for rep in range(10):
        skf = StratifiedKFold(5, shuffle=True, random_state=rep)
        oof = cross_val_predict(mdl, Xs, y, cv=skf, method="predict_proba")[:, 1]
        aps.append(average_precision_score(y, oof))
    print(f"{name}: PR-AUC {np.mean(aps):.3f} ± {np.std(aps):.3f}")
print()

# 4) test에 같은 절차 적용 (test 자기 자신으로 fit)
print("=== 4) test 점수 저장 ===")
sc_te, _ = anomaly_scores(te, lab_te)
res = pd.DataFrame(sc_te)
res.insert(0, "regime", lab_te)
res.insert(0, "row", np.arange(len(te)))
if DROP_COLS and DROP_COLS[0] in test.columns:
    res.insert(1, DROP_COLS[0], test[DROP_COLS[0]].values)
res.to_csv(OUT_PATH, index=False)
print(f"저장: {OUT_PATH} ({len(res)}행). 상위 점수 분포:")
print(res["ensemble"].describe().round(3).to_string())

사용 컬럼 23개 | train (1211, 23) 불량 17개 (1.404%) | test (35239, 23)

=== 1) 레짐 (기준 변수: Cushion_Position) ===
train 레짐 비율: [0.63336086 0.36663914] | test 레짐 비율: [0.51508272 0.48491728]
train 레짐 0: 행 767개, 불량 16개 (2.09%)
train 레짐 1: 행 444개, 불량 1개 (0.23%)
Fisher 정확검정 p = 0.0088

=== 2) 비지도 절차: train에서 검증 (기준선 PR-AUC = 불량률 0.014) ===
          PR-AUC  ROC-AUC  recall@2%  recall@5%  recall@10%
iforest    0.180    0.787      0.412      0.412       0.412
pca        0.051    0.797      0.000      0.235       0.471
mahal      0.403    0.774      0.412      0.471       0.588
ensemble   0.116    0.808      0.235      0.412       0.588
앙상블 기준 불량 17개의 순위(작을수록 잘 잡음): [8, 8, 13, 13, 25, 57, 57, 61, 75, 97, 199, 422, 458, 558, 580, 642, 764] 

=== 3) 비교: train 지도학습 (5-fold x 10회 반복 OOF PR-AUC) ===
logreg: PR-AUC 0.445 ± 0.030
extratrees: PR-AUC 0.395 ± 0.037

=== 4) test 점수 저장 ===
저장: test_anomaly_scores.csv (35239행). 상위 점수 분포:
count    35239.000
mean         0.500
std          0.245
min          0.009
25

1. 레짐 효과가 통계적으로 확인됐어. train 불량 17개 중 16개가 Cushion_Position이 낮은 레짐에 있고, 불량률이 2.09% 대 0.23%야. Fisher p는 0.0088이야. 다만 test에서 어느 쪽이 대응되는지는 확인할 수 없다는 한계는 그대로야.

2. 비지도 Mahalanobis가 지도학습에 거의 근접해. Mahalanobis의 PR-AUC가 0.403이고 지도학습 logreg는 0.445야. 라벨을 쓰지 않은 점수가 지도학습과 비슷하다는 건 이 데이터의 불량이 "특정 패턴"보다 정상 운전 구조에서 벗어난 상태에 가깝다는 뜻이야. 라벨을 쓰지 않으니 test로 옮길 수 있다는 게 가장 큰 장점이야.

3. IsolationForest는 극단값만 잡아. recall이 2%에서 10%까지 0.412로 변하지 않아. 극단값 7개만 잡고 나머지는 못 잡는 거야. Mahalanobis는 recall@10%에서 0.588까지 올라가니, 변수 하나씩 보면 정상인데 변수들의 조합이 이상한 불량 3개를 더 잡은 거야. 보고서에 쓸 만한 서사야.

4. 앙상블은 오히려 손해야. 앙상블 PR-AUC가 0.116으로 Mahalanobis 단독(0.403)보다 낮아. PCA가 노이즈를 섞은 거야. PCA는 빼고 Mahalanobis를 중심에 두는 게 좋아.

5. 놓친 불량이 7개나 돼. 순위가 199위 이하인 불량이 7개(199, 422, 458, 558, 580, 642, 764)야. 이건 비지도 점수로는 사실상 안 보이는 불량이라, 보고서의 "실패 조건 분석"의 핵심 자료가 돼.

Mahalanobis를 17개 불량 결과를 보고 골랐으니 0.403은 낙관적이야. 보고서에는 "최선의 방법을 고른 뒤의 값"이라고 밝혀야 해.
지도학습의 ±0.030은 믿으면 안 돼. 반복 CV의 분할을 바꿔도 얼마나 흔들리는지를 보여줄 뿐, 불량 17개라는 표본 자체의 불확실성은 훨씬 커. 신뢰구간은 bootstrap으로 따로 구해서 보고서에 넣어야 해.

In [17]:
"""
4차 스크립트 (제품별로 따로 실행: cn7, rg3)
  A) Mahalanobis 변형 몇 개와 IsolationForest 블렌드를 train에서 비교 (변형은 의도적으로 적게)
  B) 불량 17개(rg3는 25개) 각각의 순위·레짐·가장 튀는 변수 -> 오류분석 표
  C) train 지도모델(logreg)이 test로 옮겨질 수 있는지: 비지도 점수와의 순위 상관을
     train(OOF)과 test에서 비교
"""
import warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf, MinCovDet
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore")

# ===== 설정 =====

TARGET = "PassOrFail"
DEFECT_VALUE = 1
DROP_COLS = ["Unnamed: 0"]
REGIME_COL = "Cushion_Position"
BEST = "mahal_mcd90"   # B, C에서 사용할 기본 점수 (A 결과를 보고 바꿔도 됨)
SEED = 42
# ================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = [c for c in train.columns if c.lower() == TARGET.lower()][0]
y = (train[TARGET] == DEFECT_VALUE).astype(int).values
cols = [
    c for c in train.columns
    if c in test.columns and c != TARGET and c not in DROP_COLS
    and pd.api.types.is_numeric_dtype(train[c]) and train[c].std() > 1e-9
]
tr, te = train[cols].copy(), test[cols].copy()
print(f"사용 컬럼 {len(cols)}개 | train 불량 {y.sum()}개 | test {len(te)}행\n")


def get_regime(x):
    v = x.clip(x.quantile(0.01), x.quantile(0.99)).values.reshape(-1, 1)
    km = KMeans(2, n_init=10, random_state=SEED).fit(v)
    lab = km.labels_
    return 1 - lab if km.cluster_centers_[0, 0] > km.cluster_centers_[1, 0] else lab


def robust_norm(X, lab):
    out = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
    for r in np.unique(lab):
        m = lab == r
        sub = X[m]
        med = sub.median()
        mad = 1.4826 * (sub - med).abs().median()
        scale = mad.where(mad > 1e-9, sub.std())
        scale = scale.where(scale > 1e-9, 1.0)
        out.loc[m] = ((sub - med) / scale).values
    return out


def zr(s):
    med = np.median(s)
    mad = 1.4826 * np.median(np.abs(s - med))
    if mad < 1e-9:
        mad = s.std() if s.std() > 1e-9 else 1.0
    return (s - med) / mad


def mahal(Xn, lab, method):
    out = np.zeros(len(Xn))
    for r in np.unique(lab):
        m = lab == r
        A = Xn[m].values
        A = np.clip(A[:, A.std(axis=0) > 1e-9], -15, 15)
        try:
            if method.startswith("mcd"):
                idx = np.random.RandomState(SEED).choice(len(A), min(len(A), 5000), replace=False)
                cov = MinCovDet(support_fraction=int(method[3:]) / 100, random_state=SEED).fit(A[idx])
            else:
                cov = LedoitWolf().fit(A)
        except Exception:
            cov = LedoitWolf().fit(A)
        out[m] = zr(cov.mahalanobis(A))
    return out


def iforest(Xn, lab):
    out = np.zeros(len(Xn))
    for r in np.unique(lab):
        m = lab == r
        A = Xn[m].values
        A = A[:, A.std(axis=0) > 1e-9]
        iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(A)
        out[m] = zr(-iso.score_samples(A))
    return out


def all_scores(X, lab):
    Xn = robust_norm(X, lab)
    zero = np.zeros(len(X), dtype=int)
    Xg = robust_norm(X, zero)
    s = {
        "mahal_mcd90": mahal(Xn, lab, "mcd90"),
        "mahal_mcd75": mahal(Xn, lab, "mcd75"),
        "mahal_shrink": mahal(Xn, lab, "lw"),
        "mahal_global": mahal(Xg, zero, "mcd90"),
        "iforest": iforest(Xn, lab),
    }
    rk = lambda v: rankdata(v) / len(v)
    s["blend_mahal+if"] = (rk(s["mahal_mcd90"]) + rk(s["iforest"])) / 2
    return s, Xn


def evaluate(y, s):
    order = np.argsort(-s)
    res = {"PR-AUC": average_precision_score(y, s), "ROC-AUC": roc_auc_score(y, s)}
    for f in [0.02, 0.05, 0.10]:
        res[f"recall@{int(f * 100)}%"] = y[order[: int(len(y) * f)]].sum() / y.sum()
    return res


lab_tr, lab_te = get_regime(tr[REGIME_COL]), get_regime(te[REGIME_COL])
sc_tr, Xn_tr = all_scores(tr, lab_tr)

# A) 변형 비교
print("=== A) train에서 점수 변형 비교 (기준선 PR-AUC = %.3f) ===" % y.mean())
print(pd.DataFrame({k: evaluate(y, v) for k, v in sc_tr.items()}).T.round(3).to_string())
print("주의: 17개 불량으로 고른 값이라 낙관적임. 차이가 작은 변형끼리는 우열을 주장하지 말 것.\n")

# B) 불량별 프로파일
print("=== B) 불량 샘플 프로파일 (%s 기준) ===" % BEST)
rank = rankdata(-sc_tr[BEST])
rows = []
for i in np.where(y == 1)[0]:
    z = Xn_tr.iloc[i]
    top = z.abs().sort_values(ascending=False).head(3).index
    rows.append({
        "row": i,
        "rank": int(rank[i]),
        "regime": int(lab_tr[i]),
        "max|z|": round(float(z.abs().max()), 1),
        "top3(레짐 내 z)": ", ".join(f"{c}({z[c]:+.1f})" for c in top),
    })
prof = pd.DataFrame(rows).sort_values("rank")
prof["구분"] = np.where(prof["max|z|"] > 5, "극단값", "관계/복합 이탈 또는 미검출")
print(prof.to_string(index=False))
prof.to_csv("defect_profile.csv", index=False)
print("저장: defect_profile.csv\n")

# C) 지도모델 이식 가능성 점검
print("=== C) train 지도모델(logreg)의 test 이식 가능성 ===")
Xs_tr = np.c_[np.clip(Xn_tr.values, -15, 15), lab_tr]
Xn_te = robust_norm(te, lab_te)
Xs_te = np.c_[np.clip(Xn_te.values, -15, 15), lab_te]
lr = LogisticRegression(C=0.1, class_weight="balanced", max_iter=3000)
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
oof = cross_val_predict(lr, Xs_tr, y, cv=skf, method="decision_function")
lr.fit(Xs_tr, y)
lr_te = lr.decision_function(Xs_te)
mah_tr = sc_tr[BEST]
mah_te = mahal(Xn_te, lab_te, BEST.replace("mahal_", ""))


def top_overlap(a, b, frac=0.02):
    k = max(1, int(len(a) * frac))
    A, B = set(np.argsort(-a)[:k]), set(np.argsort(-b)[:k])
    return len(A & B) / k


rho_tr = spearmanr(oof, mah_tr).correlation
rho_te = spearmanr(lr_te, mah_te).correlation
print(f"Spearman(logreg, mahal)  train OOF: {rho_tr:.3f} | test: {rho_te:.3f}")
for r in [0, 1]:
    print(f"  test 레짐 {r}: {spearmanr(lr_te[lab_te == r], mah_te[lab_te == r]).correlation:.3f}")
print(f"상위 2% 겹침(우연 기대치 0.02)  train: {top_overlap(oof, mah_tr):.3f} | test: {top_overlap(lr_te, mah_te):.3f}")
print("해석: test 상관이 train보다 크게 낮으면 logreg가 test로 이식되지 않는 것 -> 최종 점수에서 제외")

사용 컬럼 23개 | train 불량 17개 | test 35239행

=== A) train에서 점수 변형 비교 (기준선 PR-AUC = 0.014) ===
                PR-AUC  ROC-AUC  recall@2%  recall@5%  recall@10%
mahal_mcd90      0.403    0.774      0.412      0.471       0.588
mahal_mcd75      0.268    0.780      0.412      0.412       0.529
mahal_shrink     0.178    0.812      0.412      0.412       0.529
mahal_global     0.358    0.830      0.412      0.588       0.588
iforest          0.180    0.787      0.412      0.412       0.412
blend_mahal+if   0.270    0.787      0.412      0.412       0.529
주의: 17개 불량으로 고른 값이라 낙관적임. 차이가 작은 변형끼리는 우열을 주장하지 말 것.

=== B) 불량 샘플 프로파일 (mahal_mcd90 기준) ===
 row  rank  regime  max|z|                                                                        top3(레짐 내 z)  구분
 117     1       0    11.3    Cushion_Position(-11.3), Max_Injection_Speed(+11.0), Plasticizing_Position(+5.6) 극단값
 118     1       0    11.3    Cushion_Position(-11.3), Max_Injection_Speed(+11.0), Plasticizing_Position(+5.6) 극단값
 119     3 

In [18]:
"""
5차 스크립트 (제품별로 따로 실행: cn7, rg3)
불량 행 번호가 연속·반복 패턴으로 몰려 있다는 단서를 확인한다.
  1) train이 시간순(연속 샷)인지: 이웃 행 자기상관 vs 섞은 데이터
  2) 불량 에피소드(연속 구간) 개수와 그 구간 행 출력
  3) 레짐 x 레벨(Plasticizing_Position, Clamp_Close_Time) 조합별 불량률 + test 비중
  4) test 원본 인덱스가 연속 구간(run)으로 이루어졌는지
  5) 이웃 행 평활화가 train 점수를 개선하는지 (시간순일 때만 의미 있음)
"""
import warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf, MinCovDet
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")

# ===== 설정 =====

TARGET = "PassOrFail"
DEFECT_VALUE = 1
DROP_COLS = ["Unnamed: 0"]
REGIME_COL = "Cushion_Position"
LEVEL_COLS = ["Plasticizing_Position", "Clamp_Close_Time"]
SHOW_COLS = [
    "Plasticizing_Position", "Clamp_Close_Time", "Cushion_Position",
    "Max_Injection_Speed", "Barrel_Temperature_3", "Mold_Temperature_4", "Cycle_Time",
]
EPISODE_GAP = 5     # 불량 간격이 이 값 이하이면 같은 에피소드
SEED = 42
# ================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = [c for c in train.columns if c.lower() == TARGET.lower()][0]
y = (train[TARGET] == DEFECT_VALUE).astype(int).values
cols = [
    c for c in train.columns
    if c in test.columns and c != TARGET and c not in DROP_COLS
    and pd.api.types.is_numeric_dtype(train[c]) and train[c].std() > 1e-9
]
tr, te = train[cols].copy(), test[cols].copy()


def get_regime(x):
    v = x.clip(x.quantile(0.01), x.quantile(0.99)).values.reshape(-1, 1)
    km = KMeans(2, n_init=10, random_state=SEED).fit(v)
    lab = km.labels_
    return 1 - lab if km.cluster_centers_[0, 0] > km.cluster_centers_[1, 0] else lab


# 1) 시간순 여부
print("=== 1) train 행 순서가 시간순인가 ===")
lag = np.nanmean([tr[c].autocorr(1) for c in cols])
shuf = tr.sample(frac=1, random_state=SEED).reset_index(drop=True)
lag_s = np.nanmean([shuf[c].autocorr(1) for c in cols])
print(f"이웃 행 자기상관(평균): 원본 {lag:.3f} | 섞은 데이터 {lag_s:.3f}")
print("원본이 섞은 것보다 뚜렷이 크면 시간순(연속 샷)일 가능성이 높음\n")

# 2) 불량 에피소드
print("=== 2) 불량 에피소드 ===")
pos = np.where(y == 1)[0]
eps = np.split(pos, np.where(np.diff(pos) > EPISODE_GAP)[0] + 1)
print(f"불량 {len(pos)}개 -> 에피소드 {len(eps)}개 (간격 {EPISODE_GAP}행 이하를 같은 에피소드로 묶음)")
for e in eps:
    print(f"  행 {e.min()}~{e.max()}: 불량 {len(e)}개")
big = max(eps, key=len)
lo, hi = max(0, big.min() - 3), min(len(train), big.max() + 4)
hi = min(hi, lo + 60)
show = [c for c in SHOW_COLS if c in train.columns]
print(f"\n가장 큰 에피소드 주변 (행 {lo}~{hi - 1}), {TARGET} 포함:")
print(train.iloc[lo:hi][[TARGET] + show].round(2).to_string())
print()

# 3) 레짐 x 레벨 조합별 불량률
print("=== 3) 레짐 x 레벨 조합별 불량률 (train) 과 test 비중 ===")
lab_tr, lab_te = get_regime(tr[REGIME_COL]), get_regime(te[REGIME_COL])
keys = ["regime"] + [f"{c}_lv" for c in LEVEL_COLS if c in cols]
d_tr = pd.DataFrame({"regime": lab_tr, "y": y})
d_te = pd.DataFrame({"regime": lab_te})
for c in LEVEL_COLS:
    if c in cols:
        d_tr[f"{c}_lv"] = get_regime(tr[c])
        d_te[f"{c}_lv"] = get_regime(te[c])
g = d_tr.groupby(keys).agg(train_rows=("y", "size"), defects=("y", "sum"))
g["defect_rate"] = (g.defects / g.train_rows).round(3)
g["test_share"] = (d_te.groupby(keys).size() / len(d_te)).round(3)
g["train_share"] = (g.train_rows / len(d_tr)).round(3)
print(g.to_string())
print("레벨 0/1은 각 데이터셋 안에서 낮은 값/높은 값 (두 데이터의 같은 레벨이라는 가정)\n")

# 4) test 원본 인덱스 연속성
print("=== 4) test 원본 인덱스(Unnamed: 0) 연속성 ===")
if DROP_COLS and DROP_COLS[0] in test.columns:
    idx = test[DROP_COLS[0]].values
    print(f"파일 순서대로 증가하는가: {bool((np.diff(idx) > 0).all())}")
    d = np.diff(np.sort(idx))
    runs = np.split(np.sort(idx), np.where(d > 1)[0] + 1)
    lens = np.array([len(r) for r in runs])
    print(f"gap==1 비율 {np.mean(d == 1):.3f} | 연속 구간(run) {len(runs)}개 | "
          f"run 길이 중앙값 {np.median(lens):.0f}, 최대 {lens.max()}, 최소 {lens.min()}")
    print(f"gap 분위수(50/90/99%): {np.percentile(d, [50, 90, 99]).round(0)}\n")
else:
    print("인덱스 컬럼 없음\n")


# 5) 이웃 행 평활화 효과
def robust_norm(X, lab):
    out = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
    for r in np.unique(lab):
        m = lab == r
        sub = X[m]
        med = sub.median()
        mad = 1.4826 * (sub - med).abs().median()
        scale = mad.where(mad > 1e-9, sub.std())
        scale = scale.where(scale > 1e-9, 1.0)
        out.loc[m] = ((sub - med) / scale).values
    return out


def zr(s):
    med = np.median(s)
    mad = 1.4826 * np.median(np.abs(s - med))
    if mad < 1e-9:
        mad = s.std() if s.std() > 1e-9 else 1.0
    return (s - med) / mad


def mahal(Xn, lab):
    out = np.zeros(len(Xn))
    for r in np.unique(lab):
        m = lab == r
        A = Xn[m].values
        A = np.clip(A[:, A.std(axis=0) > 1e-9], -15, 15)
        try:
            idx = np.random.RandomState(SEED).choice(len(A), min(len(A), 5000), replace=False)
            cov = MinCovDet(support_fraction=0.9, random_state=SEED).fit(A[idx])
        except Exception:
            cov = LedoitWolf().fit(A)
        out[m] = zr(cov.mahalanobis(A))
    return out


def evaluate(y, s):
    order = np.argsort(-s)
    res = {"PR-AUC": average_precision_score(y, s), "ROC-AUC": roc_auc_score(y, s)}
    for f in [0.02, 0.05, 0.10]:
        res[f"recall@{int(f * 100)}%"] = y[order[: int(len(y) * f)]].sum() / y.sum()
    return res


print("=== 5) 이웃 행 평활화 (train 행 순서 기준, 센터 윈도우) ===")
s = mahal(robust_norm(tr, lab_tr), lab_tr)
rows = {}
for w in [1, 3, 5, 9]:
    ser = pd.Series(s)
    rows[f"mean w={w}"] = evaluate(y, ser.rolling(w, center=True, min_periods=1).mean().values)
    rows[f"max  w={w}"] = evaluate(y, ser.rolling(w, center=True, min_periods=1).max().values)
print(pd.DataFrame(rows).T.round(3).to_string())
print("w=1이 평활화 없음. train이 시간순이 아니면 이 표는 의미 없음. "
      "불량이 연속 구간에 몰려 있어 개선폭이 과대평가될 수 있음.")

=== 1) train 행 순서가 시간순인가 ===
이웃 행 자기상관(평균): 원본 0.790 | 섞은 데이터 0.003
원본이 섞은 것보다 뚜렷이 크면 시간순(연속 샷)일 가능성이 높음

=== 2) 불량 에피소드 ===
불량 17개 -> 에피소드 3개 (간격 5행 이하를 같은 에피소드로 묶음)
  행 47~47: 불량 1개
  행 80~80: 불량 1개
  행 95~120: 불량 15개

가장 큰 에피소드 주변 (행 92~123), PassOrFail 포함:
     PassOrFail  Plasticizing_Position  Clamp_Close_Time  Cushion_Position  Max_Injection_Speed  Barrel_Temperature_3  Mold_Temperature_4  Cycle_Time
92            0                   1.41              0.98             -0.71                -0.57                 -1.94                1.39        0.36
93            0                   1.47              0.98             -0.71                -0.57                  0.28                1.39        0.36
94            0                   1.47              0.98             -0.71                -0.57                  0.28                1.39        0.36
95            1                   1.47              0.98              1.20                -1.35                  0.60                1.39  

In [19]:
"""
6차 스크립트 (제품별로 따로 실행: cn7, rg3)
연속 두 행이 RH/LH 한 쌍(같은 샷)이라는 사실을 데이터로 확인하고, 평가 방식을 샷 단위로 바꾼다.
  1) train 쌍 구조: 어느 위상(홀/짝)에서 두 행이 동일한가, 전체 구간에서 유지되는가
  2) 쌍 안의 불량 라벨: 양쪽 불량 / 한쪽만 불량(첫째·둘째)
  3) 샷 단위 재평가: 쌍 단위 PR-AUC / recall / precision, 행 단위와 비교
  4) test 쌍 구조와 샘플링 구조: 인접 행의 동일 비율, 인덱스 홀짝, gap 분포 vs 무작위 추출
"""
import warnings
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf, MinCovDet
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")

# ===== 설정 =====
TARGET = "PassOrFail"
DEFECT_VALUE = 1
DROP_COLS = ["Unnamed: 0"]
REGIME_COL = "Cushion_Position"
TOL = 1e-6          # 두 행이 '동일'하다고 볼 최대 차이
SEED = 42
# ================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = [c for c in train.columns if c.lower() == TARGET.lower()][0]
y = (train[TARGET] == DEFECT_VALUE).astype(int).values
cols = [
    c for c in train.columns
    if c in test.columns and c != TARGET and c not in DROP_COLS
    and pd.api.types.is_numeric_dtype(train[c]) and train[c].std() > 1e-9
]
tr, te = train[cols].copy(), test[cols].copy()
n = len(tr)

# 1) train 쌍 구조
print("=== 1) train 쌍 구조 ===")
X = tr.values
d = np.abs(X[:-1] - X[1:]).max(axis=1)          # d[i]: 행 i와 i+1의 최대 차이
same = d < TOL
print(f"인접 행이 모든 피처에서 동일한 비율: {same.mean():.3f}")
for p in [0, 1]:
    dp = d[p::2]
    print(f"  i%2=={p} 위치의 (i, i+1) 동일 비율 {np.mean(dp < TOL):.3f} | 중앙 차이 {np.median(dp):.4f}")
phase = int(np.mean(d[1::2] < TOL) > np.mean(d[0::2] < TOL))
print(f"-> 쌍 시작 위상: i%2=={phase} (이 위치의 행이 쌍의 '첫째')")
aligned = same[phase::2]
broken = (np.arange(len(same))[phase::2])[~aligned]
print(f"위상에 맞는 쌍 중 동일하지 않은 쌍 {len(broken)}개 / {len(aligned)}개; 앞 10개 위치: {broken[:10].tolist()}\n")

pair_id = (np.arange(n) - phase) // 2
is_first = (np.arange(n) % 2) == phase

# 2) 쌍 안의 불량 라벨
print("=== 2) 쌍 안의 불량 라벨 ===")
df = pd.DataFrame({"pair": pair_id, "first": is_first, "y": y})
g = df.groupby("pair")
size = g.size()
full = size[size == 2].index
sub = df[df.pair.isin(full)]
yf = sub[sub["first"]].set_index("pair")["y"]
ys = sub[~sub["first"]].set_index("pair")["y"]
both = int(((yf == 1) & (ys == 1)).sum())
only_first = int(((yf == 1) & (ys == 0)).sum())
only_second = int(((yf == 0) & (ys == 1)).sum())
print(f"완전한 쌍 {len(full)}개: 양쪽 불량 {both} | 첫째만 불량 {only_first} | 둘째만 불량 {only_second}")
print(f"쌍에 속하지 않은 행의 불량: {int(df[~df.pair.isin(full)].y.sum())}개")
print("한쪽만 불량인 쌍은 두 행의 피처가 같으므로 샷 단위 피처로는 구분 불가 -> 해당 불량의 precision 상한 50%\n")


# 점수 계산용 함수
def get_regime(x):
    v = x.clip(x.quantile(0.01), x.quantile(0.99)).values.reshape(-1, 1)
    km = KMeans(2, n_init=10, random_state=SEED).fit(v)
    lab = km.labels_
    return 1 - lab if km.cluster_centers_[0, 0] > km.cluster_centers_[1, 0] else lab


def robust_norm(Xd, lab):
    out = pd.DataFrame(index=Xd.index, columns=Xd.columns, dtype=float)
    for r in np.unique(lab):
        m = lab == r
        sub_ = Xd[m]
        med = sub_.median()
        mad = 1.4826 * (sub_ - med).abs().median()
        scale = mad.where(mad > 1e-9, sub_.std())
        scale = scale.where(scale > 1e-9, 1.0)
        out.loc[m] = ((sub_ - med) / scale).values
    return out


def zr(s):
    med = np.median(s)
    mad = 1.4826 * np.median(np.abs(s - med))
    if mad < 1e-9:
        mad = s.std() if s.std() > 1e-9 else 1.0
    return (s - med) / mad


def mahal(Xn, lab):
    out = np.zeros(len(Xn))
    for r in np.unique(lab):
        m = lab == r
        A = Xn[m].values
        A = np.clip(A[:, A.std(axis=0) > 1e-9], -15, 15)
        try:
            idx = np.random.RandomState(SEED).choice(len(A), min(len(A), 5000), replace=False)
            cov = MinCovDet(support_fraction=0.9, random_state=SEED).fit(A[idx])
        except Exception:
            cov = LedoitWolf().fit(A)
        out[m] = zr(cov.mahalanobis(A))
    return out


def evaluate(yv, s):
    order = np.argsort(-s)
    res = {"PR-AUC": average_precision_score(yv, s), "ROC-AUC": roc_auc_score(yv, s)}
    for f in [0.02, 0.05, 0.10]:
        k = max(1, int(len(yv) * f))
        hit = yv[order[:k]].sum()
        res[f"recall@{int(f * 100)}%"] = hit / yv.sum()
        res[f"precision@{int(f * 100)}%"] = hit / k
    return res


# 3) 샷 단위 재평가
print("=== 3) 행 단위 vs 샷(쌍) 단위 평가 (Mahalanobis, 레짐별 robust) ===")
lab_tr = get_regime(tr[REGIME_COL])
s = mahal(robust_norm(tr, lab_tr), lab_tr)
shot = pd.DataFrame({"pair": pair_id, "s": s, "y": y}).groupby("pair").agg(s=("s", "mean"), y=("y", "max"))
print(f"샷 수 {len(shot)}개, 불량 샷 {int(shot.y.sum())}개 (불량 행 {int(y.sum())}개)")
res = pd.DataFrame({"행 단위": evaluate(y, s), "샷 단위": evaluate(shot.y.values, shot.s.values)}).T
print(res.round(3).to_string())
print("샷 단위 점수는 한 쌍의 평균, 샷 라벨은 쌍 중 하나라도 불량이면 1. 불량 샷은 소수이므로 표본이 더 작다는 점에 유의.\n")

# 4) test 쌍 / 샘플링 구조
print("=== 4) test 쌍 구조와 샘플링 구조 ===")
if DROP_COLS and DROP_COLS[0] in test.columns:
    idx = test[DROP_COLS[0]].values
    order = np.argsort(idx)
    idx_s, Xt = idx[order], te.values[order]
    gap = np.diff(idx_s)
    eq = np.abs(Xt[:-1] - Xt[1:]).max(axis=1) < TOL
    print(f"gap==1인 인접 행 중 피처 동일 비율: {eq[gap == 1].mean():.3f} (쌍이면 1에 가까움)")
    print(f"gap>1인 인접 행 중 피처 동일 비율: {eq[gap > 1].mean():.3f}")
    pr = (gap == 1) & eq
    print(f"동일 쌍으로 확인되는 인접 행 쌍: {int(pr.sum())}개")
    par = pd.Series(idx_s[:-1][pr] % 2).value_counts(normalize=True).round(3).to_dict()
    print(f"쌍의 앞 행 원본 인덱스 홀짝 분포 {{0:짝, 1:홀}}: {par}")
    p1 = np.mean(gap == 1)
    print(f"gap==1 비율 p={p1:.3f}. gap 누적분포: 관측 vs 무작위 추출(기하분포) 기대")
    for k in range(1, 9):
        print(f"  gap<={k}: 관측 {np.mean(gap <= k):.3f} | 기대 {1 - (1 - p1) ** k:.3f}")
else:
    print("test에 원본 인덱스 컬럼 없음")

=== 1) train 쌍 구조 ===
인접 행이 모든 피처에서 동일한 비율: 0.471
  i%2==0 위치의 (i, i+1) 동일 비율 0.026 | 중앙 차이 2.2432
  i%2==1 위치의 (i, i+1) 동일 비율 0.916 | 중앙 차이 0.0000
-> 쌍 시작 위상: i%2==1 (이 위치의 행이 쌍의 '첫째')
위상에 맞는 쌍 중 동일하지 않은 쌍 51개 / 605개; 앞 10개 위치: [295, 297, 299, 309, 311, 395, 397, 401, 403, 405]

=== 2) 쌍 안의 불량 라벨 ===
완전한 쌍 605개: 양쪽 불량 3 | 첫째만 불량 10 | 둘째만 불량 1
쌍에 속하지 않은 행의 불량: 0개
한쪽만 불량인 쌍은 두 행의 피처가 같으므로 샷 단위 피처로는 구분 불가 -> 해당 불량의 precision 상한 50%

=== 3) 행 단위 vs 샷(쌍) 단위 평가 (Mahalanobis, 레짐별 robust) ===
샷 수 606개, 불량 샷 14개 (불량 행 17개)
      PR-AUC  ROC-AUC  recall@2%  precision@2%  recall@5%  precision@5%  recall@10%  precision@10%
행 단위   0.403    0.774      0.412         0.292      0.471         0.133       0.588          0.083
샷 단위   0.294    0.725      0.286         0.333      0.357         0.167       0.500          0.117
샷 단위 점수는 한 쌍의 평균, 샷 라벨은 쌍 중 하나라도 불량이면 1. 불량 샷은 소수이므로 표본이 더 작다는 점에 유의.

=== 4) test 쌍 구조와 샘플링 구조 ===
gap==1인 인접 행 중 피처 동일 비율: 0.430 (쌍이면 1에 가까움)
gap>1인 인접 행 중 피처 동일 비율: 0.000
동일 쌍으로 확

In [21]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf, MinCovDet
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIG
# ============================================================
TRAIN_PATH = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv"
TEST_PATH  = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_unlabeled_cn7.csv"

TARGET = "PassOrFail"
DEFECT_VALUE = 1
ID_COL = "Unnamed: 0"
DROP_COLS = [ID_COL]
REGIME_COL = "Cushion_Position"

SEED = 42
TOL = 1e-6
ROLL_WINDOW = 5

# shot 단위 CV: pair 누수는 제거하지만 시간 인접성 누수는 일부 남을 수 있음
SHOT_CV_SPLITS = 5
SHOT_CV_REPEATS = 5

# 주 검증: 연속 5 shot을 하나의 block으로 묶음
# block 안 샘플은 train/val로 갈라지지 않음
BLOCK_SIZE = 5
BLOCK_CV_SPLITS = 3
PURGE_BLOCKS = 1

# strict forward diagnostic
FORWARD_BLOCKS = 6
FORWARD_PURGE_SHOTS = 5

OUT_DIR = Path("cn7_7th_outputs")
OUT_DIR.mkdir(exist_ok=True)


# ============================================================
# 1. UTIL
# ============================================================
def find_case_insensitive(columns, wanted):
    hit = [c for c in columns if c.lower() == wanted.lower()]
    if not hit:
        raise KeyError(f"'{wanted}' 컬럼을 찾지 못했습니다.\ncolumns={list(columns)}")
    return hit[0]


def safe_roc_auc(y, s):
    y = np.asarray(y)
    s = np.asarray(s)
    return roc_auc_score(y, s) if np.unique(y).size == 2 else np.nan


def evaluate_scores(y, score):
    y = np.asarray(y).astype(int)
    score = np.asarray(score, dtype=float)
    ok = np.isfinite(score)
    y = y[ok]
    score = score[ok]

    if len(y) == 0 or y.sum() == 0:
        return {
            "n": len(y), "defects": int(y.sum()),
            "PR_AUC": np.nan, "ROC_AUC": np.nan,
            "recall@2%": np.nan, "precision@2%": np.nan,
            "recall@5%": np.nan, "precision@5%": np.nan,
            "recall@10%": np.nan, "precision@10%": np.nan,
        }

    out = {
        "n": len(y),
        "defects": int(y.sum()),
        "PR_AUC": average_precision_score(y, score),
        "ROC_AUC": safe_roc_auc(y, score),
    }
    order = np.argsort(-score)
    for frac in [0.02, 0.05, 0.10]:
        k = max(1, int(np.ceil(len(y) * frac)))
        hit = int(y[order[:k]].sum())
        out[f"recall@{int(frac*100)}%"] = hit / y.sum()
        out[f"precision@{int(frac*100)}%"] = hit / k
    return out


def pct_rank_frame(X):
    """각 데이터셋/각 fold 내부 percentile rank. 독립 StandardScaler에 덜 민감."""
    X = X.replace([np.inf, -np.inf], np.nan).copy()
    out = X.rank(method="average", pct=True)
    return out.fillna(0.5)


def robust_z_by_reference(train_values, values):
    train_values = np.asarray(train_values, dtype=float)
    values = np.asarray(values, dtype=float)
    med = np.nanmedian(train_values)
    mad = 1.4826 * np.nanmedian(np.abs(train_values - med))
    if not np.isfinite(mad) or mad < 1e-9:
        sd = np.nanstd(train_values)
        mad = sd if np.isfinite(sd) and sd > 1e-9 else 1.0
    return (values - med) / mad


# ============================================================
# 2. LOAD
# ============================================================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = find_case_insensitive(train.columns, TARGET)

y_row = (train[TARGET] == DEFECT_VALUE).astype(int).to_numpy()

sensor_cols = [
    c for c in train.columns
    if c in test.columns
    and c != TARGET
    and c not in DROP_COLS
    and pd.api.types.is_numeric_dtype(train[c])
    and pd.api.types.is_numeric_dtype(test[c])
    and train[c].std() > 1e-9
]

if REGIME_COL not in sensor_cols:
    raise KeyError(f"REGIME_COL={REGIME_COL} 이 sensor_cols에 없습니다.")

print("=" * 90)
print("CN7 7차 실험")
print("=" * 90)
print(f"train: {train.shape} | test: {test.shape}")
print(f"target: {TARGET} | 불량 {y_row.sum()} / {len(y_row)} = {y_row.mean():.3%}")
print(f"사용 센서: {len(sensor_cols)}개")
print(sensor_cols)
print()


# ============================================================
# 3. PAIR / SHOT 구조 추정
# ============================================================
def infer_pair_structure(df, feature_cols, id_col=None, tol=1e-6):
    X = df[feature_cols].to_numpy(dtype=float)
    d = np.nanmax(np.abs(X[:-1] - X[1:]), axis=1)
    same = d < tol

    p0 = float(np.mean(same[0::2])) if len(same[0::2]) else 0.0
    p1 = float(np.mean(same[1::2])) if len(same[1::2]) else 0.0
    phase = int(p1 > p0)

    id_parity = None
    if id_col is not None and id_col in df.columns:
        ids = pd.to_numeric(df[id_col], errors="coerce").to_numpy()
        starts = np.arange(len(df) - 1)
        m = (starts % 2 == phase) & same & np.isfinite(ids[:-1]) & np.isfinite(ids[1:]) & (np.diff(ids) == 1)
        if m.sum() > 0:
            parity_vals = (ids[:-1][m].astype(np.int64) % 2)
            counts = pd.Series(parity_vals).value_counts()
            id_parity = int(counts.index[0])

    print("=== Pair 구조 ===")
    print(f"전체 인접행 완전 동일 비율: {same.mean():.3f}")
    print(f"i%2==0 pair 동일 비율: {p0:.3f}")
    print(f"i%2==1 pair 동일 비율: {p1:.3f}")
    print(f"파일 행 기준 pair 시작 phase: {phase}")
    if id_parity is not None:
        print(f"원본 {id_col} 기준 pair 첫 행 parity: {id_parity} (0=짝수, 1=홀수)")
    else:
        print("원본 ID parity는 확정하지 못함 -> train은 행 위치 pair, test는 singleton fallback")
    print()

    return {
        "phase": phase,
        "id_parity": id_parity,
        "same": same,
        "distance": d,
    }


def shot_id_from_position(n, phase):
    out = np.empty(n, dtype=np.int64)
    sid = 0
    i = 0
    if phase == 1 and n > 0:
        out[0] = sid
        sid += 1
        i = 1
    while i < n:
        out[i] = sid
        if i + 1 < n:
            out[i + 1] = sid
        sid += 1
        i += 2
    return out


def make_shot_ids(df, pair_info, is_train):
    # 원본 row index가 있고 parity까지 확정되면 이것이 가장 안전함
    if ID_COL in df.columns and pair_info["id_parity"] is not None:
        ids = pd.to_numeric(df[ID_COL], errors="coerce")
        if ids.notna().all():
            ids = ids.astype(np.int64).to_numpy()
            return np.floor_divide(ids - pair_info["id_parity"], 2).astype(np.int64)

    if is_train:
        return shot_id_from_position(len(df), pair_info["phase"])

    # test는 중간 row가 빠져 있을 수 있으므로 위치로 억지 pair를 만들면 위험함
    # ID를 못 쓰는 경우 각 row를 독립 shot으로 둠
    return np.arange(len(df), dtype=np.int64)


pair_info = infer_pair_structure(train, sensor_cols, ID_COL, TOL)
train_shot_id = make_shot_ids(train, pair_info, is_train=True)
test_shot_id = make_shot_ids(test, pair_info, is_train=False)


# ============================================================
# 4. ROW -> SHOT 집계
# ============================================================
def aggregate_to_shot(df, shot_ids, feature_cols, target=None):
    tmp = df[feature_cols].copy()
    tmp["__shot_id"] = shot_ids
    tmp["__row_pos"] = np.arange(len(df))

    if target is not None:
        tmp["__y"] = target

    g = tmp.groupby("__shot_id", sort=True)

    mean_x = g[feature_cols].mean()
    max_x = g[feature_cols].max()
    min_x = g[feature_cols].min()
    ranges = max_x - min_x

    shot = mean_x.copy()
    shot["pair_nrows"] = g.size().astype(float)
    shot["pair_max_range"] = ranges.max(axis=1)
    shot["pair_mean_range"] = ranges.mean(axis=1)
    shot["first_row_pos"] = g["__row_pos"].min().astype(int)

    if target is not None:
        shot["y"] = g["__y"].max().astype(int)

    # index 자체가 원본 시간/shot 순서를 근사
    shot["shot_order"] = shot.index.astype(np.int64)

    row_map = pd.DataFrame({
        "row_pos": np.arange(len(df)),
        "shot_id": shot_ids,
    })
    return shot.sort_values("shot_order"), row_map


shot_train, train_row_map = aggregate_to_shot(train, train_shot_id, sensor_cols, y_row)
shot_test, test_row_map = aggregate_to_shot(test, test_shot_id, sensor_cols, target=None)

y_shot = shot_train["y"].to_numpy(dtype=int)

print("=== Shot 집계 결과 ===")
print(f"train row {len(train)} -> shot {len(shot_train)}")
print(f"불량 row {y_row.sum()} -> 불량 shot {y_shot.sum()}")
print("train shot 내 row 수:", shot_train["pair_nrows"].value_counts().sort_index().to_dict())
print("test shot 내 row 수:", shot_test["pair_nrows"].value_counts().sort_index().to_dict())
print()

# pair label 구조
pair_label = pd.DataFrame({"shot_id": train_shot_id, "y": y_row}).groupby("shot_id")["y"].agg(["size", "sum"])
print("=== Shot label 구조 ===")
print("양쪽/복수 row가 모두 불량인 shot:", int(((pair_label["size"] >= 2) & (pair_label["sum"] >= 2)).sum()))
print("한쪽만 불량인 2-row shot:", int(((pair_label["size"] == 2) & (pair_label["sum"] == 1)).sum()))
print("singleton 불량 shot:", int(((pair_label["size"] == 1) & (pair_label["sum"] == 1)).sum()))
print()


# ============================================================
# 5. CAUSAL TEMPORAL FEATURE
# ============================================================
def add_temporal_features(shot_df, base_sensor_cols, roll=5):
    out = shot_df.copy().sort_values("shot_order")
    order = out["shot_order"].to_numpy()

    gap = pd.Series(order, index=out.index).diff()
    run_start = gap.ne(1)
    run_id = run_start.cumsum().astype(int)

    out["shot_gap"] = gap.fillna(0).astype(float)
    out["is_consecutive"] = gap.eq(1).astype(int)
    out["run_id"] = run_id.to_numpy()

    # 미래를 보지 않도록 현재 shot보다 이전 shot만 rolling 기준에 사용
    for _, idx in out.groupby("run_id", sort=False).groups.items():
        idx = list(idx)
        sub = out.loc[idx, base_sensor_cols]

        prev = sub.shift(1)
        past = sub.shift(1)
        rmean = past.rolling(roll, min_periods=2).mean()
        rstd = past.rolling(roll, min_periods=2).std(ddof=0)

        delta = sub - prev
        dev = sub - rmean
        zdev = dev / (rstd + 1e-6)

        for c in base_sensor_cols:
            out.loc[idx, f"d1__{c}"] = delta[c].to_numpy()
            out.loc[idx, f"dev{roll}__{c}"] = dev[c].to_numpy()
            out.loc[idx, f"zdev{roll}__{c}"] = zdev[c].to_numpy()

    return out


shot_train_f = add_temporal_features(shot_train, sensor_cols, ROLL_WINDOW)
shot_test_f = add_temporal_features(shot_test, sensor_cols, ROLL_WINDOW)

pair_meta_cols = ["pair_nrows", "pair_max_range", "pair_mean_range", "shot_gap", "is_consecutive"]
base_features = sensor_cols + pair_meta_cols

temporal_cols = []
for c in sensor_cols:
    temporal_cols += [f"d1__{c}", f"dev{ROLL_WINDOW}__{c}", f"zdev{ROLL_WINDOW}__{c}"]

temporal_features = base_features + temporal_cols

print("=== Feature set ===")
print(f"BASE: {len(base_features)}개")
print(f"TEMPORAL: {len(temporal_features)}개")
print()


# ============================================================
# 6. CV SPLIT
# ============================================================
def make_shot_cv_splits(y):
    cv = RepeatedStratifiedKFold(
        n_splits=SHOT_CV_SPLITS,
        n_repeats=SHOT_CV_REPEATS,
        random_state=SEED,
    )
    dummy = np.zeros((len(y), 1))
    return [(tr, va) for tr, va in cv.split(dummy, y)]


def make_block_cv_splits(y, block_size=5, n_splits=3, purge_blocks=1):
    n = len(y)
    block_id = np.arange(n) // block_size

    block_stats = pd.DataFrame({"block": block_id, "y": y}).groupby("block").agg(
        rows=("y", "size"), defects=("y", "sum")
    )
    block_stats["defect_rate"] = block_stats["defects"] / block_stats["rows"]
    block_stats.to_csv(OUT_DIR / "block_stats.csv")

    print("=== 연속 block의 불량 분포 ===")
    print(block_stats[block_stats["defects"] > 0].to_string())
    print()

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    raw_splits = list(cv.split(np.zeros((n, 1)), y, groups=block_id))

    splits = []
    for fold, (tr_idx, va_idx) in enumerate(raw_splits, 1):
        val_blocks = np.unique(block_id[va_idx])
        forbidden = set()
        for b in val_blocks:
            for d in range(-purge_blocks, purge_blocks + 1):
                forbidden.add(int(b + d))

        keep = np.array([block_id[i] not in forbidden for i in tr_idx])
        tr_idx = tr_idx[keep]

        tr_pos = int(y[tr_idx].sum())
        va_pos = int(y[va_idx].sum())
        print(
            f"Block fold {fold}: train={len(tr_idx)} pos={tr_pos} | "
            f"val={len(va_idx)} pos={va_pos} | val blocks={len(val_blocks)}"
        )
        if len(np.unique(y[tr_idx])) < 2 or len(np.unique(y[va_idx])) < 2:
            print("  -> 이 fold는 한쪽 class가 없어 모델 평가에서 제외")
            continue
        splits.append((tr_idx, va_idx))
    print()
    return splits, block_id


def make_forward_splits(y, n_blocks=6, purge_shots=5):
    n = len(y)
    edges = np.linspace(0, n, n_blocks + 1, dtype=int)
    splits = []

    print("=== Strict forward CV 가능성 진단 ===")
    for b in range(n_blocks):
        lo, hi = edges[b], edges[b + 1]
        print(f"time block {b}: shot {lo:4d}~{hi-1:4d} | rows={hi-lo:4d} | defects={int(y[lo:hi].sum())}")

    print("\nForward folds:")
    for b in range(1, n_blocks):
        va = np.arange(edges[b], edges[b + 1])
        train_end = max(0, edges[b] - purge_shots)
        tr = np.arange(0, train_end)

        ok = (
            len(tr) > 0
            and len(np.unique(y[tr])) == 2
            and len(np.unique(y[va])) == 2
        )
        print(
            f"fold {b}: train 0~{train_end-1} pos={int(y[tr].sum()) if len(tr) else 0} | "
            f"val {edges[b]}~{edges[b+1]-1} pos={int(y[va].sum())} | valid={ok}"
        )
        if ok:
            splits.append((tr, va))
    print()
    return splits


shot_splits = make_shot_cv_splits(y_shot)
block_splits, block_id = make_block_cv_splits(
    y_shot, BLOCK_SIZE, BLOCK_CV_SPLITS, PURGE_BLOCKS
)
forward_splits = make_forward_splits(
    y_shot, FORWARD_BLOCKS, FORWARD_PURGE_SHOTS
)


# ============================================================
# 7. SUPERVISED MODEL CV
# ============================================================
def make_model(name):
    if name == "logreg":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.1,
                class_weight="balanced",
                max_iter=5000,
                random_state=SEED,
            )),
        ])

    if name == "extratrees":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=500,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=SEED,
            )),
        ])

    raise ValueError(name)


def transform_fold(Xtr, Xva, mode):
    if mode == "raw":
        return Xtr.copy(), Xva.copy()
    if mode == "rank":
        # train fold와 val fold를 각각 자기 분포 안에서 rank화
        # test도 자기 분포 안에서 rank화할 것이므로 독립 표준화 shift에 대응
        return pct_rank_frame(Xtr), pct_rank_frame(Xva)
    raise ValueError(mode)


def cv_predict_supervised(X, y, splits, model_name, mode):
    X = X.copy()
    y = np.asarray(y).astype(int)
    pred_sum = np.zeros(len(y), dtype=float)
    pred_n = np.zeros(len(y), dtype=int)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(splits, 1):
        if len(np.unique(y[tr_idx])) < 2 or len(np.unique(y[va_idx])) < 2:
            continue

        Xtr, Xva = transform_fold(X.iloc[tr_idx], X.iloc[va_idx], mode)
        mdl = make_model(model_name)
        mdl.fit(Xtr, y[tr_idx])
        p = mdl.predict_proba(Xva)[:, 1]

        pred_sum[va_idx] += p
        pred_n[va_idx] += 1

        m = evaluate_scores(y[va_idx], p)
        m["fold"] = fold
        fold_rows.append(m)

    oof = np.full(len(y), np.nan)
    mask = pred_n > 0
    oof[mask] = pred_sum[mask] / pred_n[mask]

    overall = evaluate_scores(y[mask], oof[mask])
    if fold_rows:
        overall["fold_PR_mean"] = float(np.nanmean([r["PR_AUC"] for r in fold_rows]))
        overall["fold_PR_min"] = float(np.nanmin([r["PR_AUC"] for r in fold_rows]))
        overall["fold_PR_max"] = float(np.nanmax([r["PR_AUC"] for r in fold_rows]))
        overall["n_folds"] = len(fold_rows)
    else:
        overall["fold_PR_mean"] = np.nan
        overall["fold_PR_min"] = np.nan
        overall["fold_PR_max"] = np.nan
        overall["n_folds"] = 0

    return oof, overall, pd.DataFrame(fold_rows)


EXPERIMENTS = {
    # 기존 scaled 값 그대로: train 내부 성능 참고용
    "base_raw": (base_features, "raw"),
    # 독립 StandardScaler shift 대응 후보
    "base_rank": (base_features, "rank"),
    # shot + 시간 변화량까지 포함한 핵심 후보
    "temporal_rank": (temporal_features, "rank"),
}

CV_SCHEMES = {
    "shot_repeated": shot_splits,
    "block_purged": block_splits,
}
if len(forward_splits) > 0:
    CV_SCHEMES["strict_forward"] = forward_splits

results = []
oof_store = {}

for cv_name, splits in CV_SCHEMES.items():
    if len(splits) == 0:
        continue
    print("=" * 90)
    print(f"SUPERVISED CV: {cv_name}")
    print("=" * 90)

    for exp_name, (features, mode) in EXPERIMENTS.items():
        X = shot_train_f[features].replace([np.inf, -np.inf], np.nan)
        for model_name in ["logreg", "extratrees"]:
            oof, met, fold_df = cv_predict_supervised(X, y_shot, splits, model_name, mode)
            key = f"{cv_name}__{exp_name}__{model_name}"
            oof_store[key] = oof

            row = {
                "cv": cv_name,
                "experiment": exp_name,
                "model": model_name,
                "mode": mode,
                **met,
            }
            results.append(row)

            print(
                f"{exp_name:14s} | {model_name:10s} | "
                f"OOF PR={met['PR_AUC']:.4f} | ROC={met['ROC_AUC']:.4f} | "
                f"fold PR={met['fold_PR_mean']:.4f} "
                f"[{met['fold_PR_min']:.4f}, {met['fold_PR_max']:.4f}]"
            )
    print()


# ============================================================
# 8. FAIR OOF MAHALANOBIS
#    fold train으로만 regime/normalizer/covariance fit -> val score
# ============================================================
def fit_regime(train_series):
    x = pd.Series(train_series).astype(float)
    lo, hi = x.quantile([0.01, 0.99]).values
    v = x.clip(lo, hi).to_numpy().reshape(-1, 1)
    km = KMeans(n_clusters=2, n_init=20, random_state=SEED).fit(v)
    centers = km.cluster_centers_.ravel()
    order = np.argsort(centers)
    remap = {int(order[0]): 0, int(order[1]): 1}
    return km, remap, lo, hi


def predict_regime(km, remap, lo, hi, series):
    x = pd.Series(series).astype(float).clip(lo, hi).to_numpy().reshape(-1, 1)
    raw = km.predict(x)
    return np.array([remap[int(v)] for v in raw], dtype=int)


def fit_regime_robust_norm(Xtr, lab_tr):
    params = {}
    for r in np.unique(lab_tr):
        sub = Xtr.loc[lab_tr == r]
        med = sub.median()
        mad = 1.4826 * (sub - med).abs().median()
        sd = sub.std()
        scale = mad.where(mad > 1e-9, sd)
        scale = scale.where(scale > 1e-9, 1.0).fillna(1.0)
        params[int(r)] = (med, scale)
    return params


def apply_regime_robust_norm(X, lab, params):
    out = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
    for r in np.unique(lab):
        med, scale = params[int(r)]
        m = lab == r
        out.loc[m] = ((X.loc[m] - med) / scale).to_numpy()
    return out


def mahal_fold_scores(Xtr, Xva, tr_regime, va_regime):
    scores = np.full(len(Xva), np.nan, dtype=float)

    for r in np.unique(tr_regime):
        mt = tr_regime == r
        mv = va_regime == r
        if mv.sum() == 0:
            continue

        A = Xtr.loc[mt].to_numpy(dtype=float)
        B = Xva.loc[mv].to_numpy(dtype=float)

        good = np.nanstd(A, axis=0) > 1e-9
        A = A[:, good]
        B = B[:, good]

        A = np.nan_to_num(A, nan=0.0, posinf=15.0, neginf=-15.0)
        B = np.nan_to_num(B, nan=0.0, posinf=15.0, neginf=-15.0)
        A = np.clip(A, -15, 15)
        B = np.clip(B, -15, 15)

        if A.shape[1] == 0:
            continue

        try:
            cov = MinCovDet(
                support_fraction=0.90,
                random_state=SEED,
            ).fit(A)
        except Exception:
            cov = LedoitWolf().fit(A)

        d_tr = cov.mahalanobis(A)
        d_va = cov.mahalanobis(B)
        z_va = robust_z_by_reference(d_tr, d_va)
        scores[np.where(mv)[0]] = z_va

    return scores


def cv_predict_mahal(X, y, splits):
    X = X.copy().reset_index(drop=True)
    y = np.asarray(y).astype(int)
    pred_sum = np.zeros(len(y), dtype=float)
    pred_n = np.zeros(len(y), dtype=int)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(splits, 1):
        if len(np.unique(y[va_idx])) < 2:
            continue

        Xtr_raw = X.iloc[tr_idx].copy()
        Xva_raw = X.iloc[va_idx].copy()

        km, remap, lo, hi = fit_regime(Xtr_raw[REGIME_COL])
        lab_tr = predict_regime(km, remap, lo, hi, Xtr_raw[REGIME_COL])
        lab_va = predict_regime(km, remap, lo, hi, Xva_raw[REGIME_COL])

        params = fit_regime_robust_norm(Xtr_raw, lab_tr)
        Xtr_n = apply_regime_robust_norm(Xtr_raw, lab_tr, params)
        Xva_n = apply_regime_robust_norm(Xva_raw, lab_va, params)

        p = mahal_fold_scores(Xtr_n, Xva_n, lab_tr, lab_va)
        ok = np.isfinite(p)
        if ok.sum() == 0:
            continue

        idx = va_idx[ok]
        pred_sum[idx] += p[ok]
        pred_n[idx] += 1

        met = evaluate_scores(y[idx], p[ok])
        met["fold"] = fold
        fold_rows.append(met)

    oof = np.full(len(y), np.nan)
    m = pred_n > 0
    oof[m] = pred_sum[m] / pred_n[m]
    overall = evaluate_scores(y[m], oof[m])
    if fold_rows:
        overall["fold_PR_mean"] = float(np.nanmean([r["PR_AUC"] for r in fold_rows]))
        overall["fold_PR_min"] = float(np.nanmin([r["PR_AUC"] for r in fold_rows]))
        overall["fold_PR_max"] = float(np.nanmax([r["PR_AUC"] for r in fold_rows]))
        overall["n_folds"] = len(fold_rows)
    else:
        overall["fold_PR_mean"] = np.nan
        overall["fold_PR_min"] = np.nan
        overall["fold_PR_max"] = np.nan
        overall["n_folds"] = 0
    return oof, overall


X_mahal = shot_train_f[sensor_cols].reset_index(drop=True)
for cv_name, splits in CV_SCHEMES.items():
    if len(splits) == 0:
        continue
    oof, met = cv_predict_mahal(X_mahal, y_shot, splits)
    key = f"{cv_name}__mahal_oof"
    oof_store[key] = oof
    results.append({
        "cv": cv_name,
        "experiment": "base_regime_robust",
        "model": "mahal_oof",
        "mode": "raw",
        **met,
    })
    print(
        f"[Mahalanobis] {cv_name:14s} | OOF PR={met['PR_AUC']:.4f} | "
        f"ROC={met['ROC_AUC']:.4f} | folds={met['n_folds']}"
    )
print()


# ============================================================
# 9. SHOT OOF -> ROW OOF 로 다시 확장하여 row 성능도 확인
# ============================================================
def expand_shot_score_to_rows(shot_index, shot_scores, row_map):
    s = pd.Series(shot_scores, index=shot_index)
    return row_map["shot_id"].map(s).to_numpy(dtype=float)


row_metric_rows = []
for key, oof in oof_store.items():
    row_score = expand_shot_score_to_rows(shot_train_f.index, oof, train_row_map)
    met = evaluate_scores(y_row, row_score)
    row_metric_rows.append({"key": key, **met})

row_metrics = pd.DataFrame(row_metric_rows).sort_values("PR_AUC", ascending=False)
print("=" * 90)
print("SHOT OOF를 ROW로 확장했을 때")
print("=" * 90)
print(row_metrics[["key", "PR_AUC", "ROC_AUC", "recall@5%", "precision@5%"]].round(4).to_string(index=False))
print()


# ============================================================
# 10. RESULT SUMMARY SAVE
# ============================================================
result_df = pd.DataFrame(results)
result_df = result_df.sort_values(["cv", "PR_AUC"], ascending=[True, False])

cols_show = [
    "cv", "experiment", "model", "mode",
    "n", "defects", "PR_AUC", "ROC_AUC",
    "recall@2%", "precision@2%",
    "recall@5%", "precision@5%",
    "recall@10%", "precision@10%",
    "fold_PR_mean", "fold_PR_min", "fold_PR_max", "n_folds",
]
cols_show = [c for c in cols_show if c in result_df.columns]

print("=" * 90)
print("최종 CV 요약")
print("=" * 90)
print(result_df[cols_show].round(4).to_string(index=False))
print()

result_df.to_csv(OUT_DIR / "cv_summary.csv", index=False)
row_metrics.to_csv(OUT_DIR / "row_level_oof_summary.csv", index=False)

# OOF score 저장
oof_df = pd.DataFrame({
    "shot_id": shot_train_f.index,
    "y": y_shot,
    "block_id": block_id,
})
for k, v in oof_store.items():
    oof_df[k] = v
oof_df.to_csv(OUT_DIR / "shot_oof_scores.csv", index=False)

# train shot feature 저장
shot_train_f.to_csv(OUT_DIR / "train_shot_features.csv")
shot_test_f.to_csv(OUT_DIR / "test_shot_features.csv")


# ============================================================
# 11. TEST SCORE 후보 생성
#     최종 제출값을 자동 선택하지 않고 후보 score만 저장
# ============================================================
def fit_full_predict_rank(train_X, test_X, y, model_name):
    Xtr = pct_rank_frame(train_X)
    Xte = pct_rank_frame(test_X)
    mdl = make_model(model_name)
    mdl.fit(Xtr, y)
    return mdl.predict_proba(Xte)[:, 1]


def transductive_mahal(X):
    """
    test 자기 분포에서 regime/robust normalization/covariance를 fit한 뒤 자기 anomaly score 산출.
    label은 전혀 사용하지 않음. 기존 3~6차의 '절차를 옮긴다' 아이디어를 shot 단위로 적용.
    """
    X = X.copy().reset_index(drop=True)
    km, remap, lo, hi = fit_regime(X[REGIME_COL])
    lab = predict_regime(km, remap, lo, hi, X[REGIME_COL])
    params = fit_regime_robust_norm(X, lab)
    Xn = apply_regime_robust_norm(X, lab, params)

    out = np.full(len(X), np.nan, dtype=float)
    for r in np.unique(lab):
        m = lab == r
        A = Xn.loc[m].to_numpy(dtype=float)
        good = np.nanstd(A, axis=0) > 1e-9
        A = A[:, good]
        A = np.nan_to_num(A, nan=0.0, posinf=15.0, neginf=-15.0)
        A = np.clip(A, -15, 15)
        if A.shape[1] == 0:
            continue
        try:
            cov = MinCovDet(support_fraction=0.90, random_state=SEED).fit(A)
        except Exception:
            cov = LedoitWolf().fit(A)
        d = cov.mahalanobis(A)
        out[m] = robust_z_by_reference(d, d)
    return out


Xtr_final = shot_train_f[temporal_features].replace([np.inf, -np.inf], np.nan)
Xte_final = shot_test_f[temporal_features].replace([np.inf, -np.inf], np.nan)

p_lr_test = fit_full_predict_rank(Xtr_final, Xte_final, y_shot, "logreg")
p_et_test = fit_full_predict_rank(Xtr_final, Xte_final, y_shot, "extratrees")
p_mah_test = transductive_mahal(shot_test_f[sensor_cols])

# shot score -> 원래 test row score
shot_lr = pd.Series(p_lr_test, index=shot_test_f.index)
shot_et = pd.Series(p_et_test, index=shot_test_f.index)
shot_mah = pd.Series(p_mah_test, index=shot_test_f.index)

out_test = pd.DataFrame({
    "row": np.arange(len(test)),
    "shot_id": test_shot_id,
    "logreg_rank_temporal": pd.Series(test_shot_id).map(shot_lr).to_numpy(),
    "extratrees_rank_temporal": pd.Series(test_shot_id).map(shot_et).to_numpy(),
    "mahal_transductive": pd.Series(test_shot_id).map(shot_mah).to_numpy(),
})
if ID_COL in test.columns:
    out_test.insert(1, ID_COL, test[ID_COL].to_numpy())

# 서로 scale이 다른 score 비교용 percentile rank
for c in ["logreg_rank_temporal", "extratrees_rank_temporal", "mahal_transductive"]:
    s = out_test[c].to_numpy(dtype=float)
    ok = np.isfinite(s)
    rr = np.full(len(s), np.nan)
    rr[ok] = rankdata(s[ok], method="average") / ok.sum()
    out_test[c + "__rank"] = rr

out_test.to_csv(OUT_DIR / "test_candidate_scores.csv", index=False)

print("저장 완료:")
for p in [
    OUT_DIR / "cv_summary.csv",
    OUT_DIR / "row_level_oof_summary.csv",
    OUT_DIR / "shot_oof_scores.csv",
    OUT_DIR / "train_shot_features.csv",
    OUT_DIR / "test_shot_features.csv",
    OUT_DIR / "test_candidate_scores.csv",
    OUT_DIR / "block_stats.csv",
]:
    print(" -", p)

print("\n중요: test_candidate_scores.csv의 세 점수는 아직 최종 앙상블하지 마세요.")
print("먼저 cv_summary.csv 결과를 보고 block_purged 성능이 유지되는지 확인해야 합니다.")


CN7 7차 실험
train: (1211, 26) | test: (35239, 25)
target: PassOrFail | 불량 17 / 1211 = 1.404%
사용 센서: 23개
['Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Plasticizing_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']

=== Pair 구조 ===
전체 인접행 완전 동일 비율: 0.471
i%2==0 pair 동일 비율: 0.026
i%2==1 pair 동일 비율: 0.916
파일 행 기준 pair 시작 phase: 1
원본 Unnamed: 0 기준 pair 첫 행 parity: 1 (0=짝수, 1=홀수)

=== Shot 집계 결과 ===
train row 1211 -> shot 606
불량 row 17 -> 불량 shot 14
train shot 내 row 수: {1.0: 1, 2.0: 605}
test shot 내 row 수: {1.0: 20177, 2.0: 7531}

=== Shot label 구조 ===
양쪽/복수 row가 모두 불량인 shot: 3
한쪽만 불량인 2-row shot: 11
sin